# Phase B v3: FinBERT 언어 확장 (Chinese + Korean)

**목적**: v2에서 영어만 의미 있는 FinBERT 점수를 얻었기 때문에, CJK(중국어/한국어) 뉴스에 대해서는 각 언어에 특화된 금융 BERT를 별도로 적용.

**진단 결과 (v5 기준)**
- Chinese: 28,084행 → std 0.032 (사실상 무신호)
- Korean : 8,703행 → std 0.101 (약신호)
- English: 4,801행 → std 0.466 (강신호, 정상 작동)

**사용 모델 (HF Hub에서 검증됨)**
- `yiyanghkust/finbert-tone-chinese` — 377K downloads, Yiyang Kust (ProsusAI finbert-tone 저자), Apache-2.0
- `snunlp/KR-FinBert-SC` — 1.5M downloads, SNU NLP Lab, Korean financial sentiment classification

**입력 파일** (로컬 `scripts/finbert_extract_zh_ko.py`로 생성)
- `data/processed/finbert_input_zh.parquet` : 16,010 unique Chinese titles
- `data/processed/finbert_input_ko.parquet` :  8,112 unique Korean titles

**출력** (Colab)
- `data/processed/finbert_results_zh.parquet`
- `data/processed/finbert_results_ko.parquet`

**예상 시간**: T4 GPU 기준 합계 5분 미만.

**주의**: 모델마다 `id2label` 순서가 다르므로 하드코딩 금지 — 동적 매핑 사용.

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/nabi_hyoghaw'  # ← 본인 경로로 수정
os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())

In [ ]:
!pip install -q transformers torch pandas pyarrow tqdm
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. 공용 배치 추론 함수

`id2label`을 동적으로 읽어 pos/neg/neu 인덱스를 찾습니다. 모델마다 라벨 순서가 다르므로 반드시 필요.

In [ ]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def find_label_indices(id2label):
    """id2label에서 pos/neg/neu 인덱스를 추출. 다양한 표기 허용."""
    mapping = {i: str(v).lower().strip() for i, v in id2label.items()}
    pos_keywords = {'positive', 'pos', 'bullish', 'good', 'label_2', '正面', '긍정'}
    neg_keywords = {'negative', 'neg', 'bearish', 'bad',  'label_0', '负面', '부정'}
    neu_keywords = {'neutral',  'neu', 'flat', 'label_1', '中性', '중립'}
    pi = ni = ui = None
    for i, v in mapping.items():
        if any(k in v for k in pos_keywords) and pi is None: pi = i
        elif any(k in v for k in neg_keywords) and ni is None: ni = i
        elif any(k in v for k in neu_keywords) and ui is None: ui = i
    return pi, ni, ui, mapping

def run_finbert(model_name, input_path, output_path, batch_size=64, max_len=256):
    print(f'\n{"="*70}\n[{model_name}]\n{"="*70}')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).eval().to(device)
    pi, ni, ui, mapping = find_label_indices(model.config.id2label)
    print(f'  id2label   : {mapping}')
    print(f'  pos_idx={pi}, neg_idx={ni}, neu_idx={ui}')
    if pi is None or ni is None:
        raise RuntimeError(f'Could not infer positive/negative indices from id2label={mapping}')

    df = pd.read_parquet(input_path)
    print(f'  input rows : {len(df):,}  from {input_path}')

    texts = df['title'].astype(str).tolist()
    n = len(texts)
    neg = np.zeros(n, dtype=np.float32)
    neu = np.zeros(n, dtype=np.float32)
    pos = np.zeros(n, dtype=np.float32)

    for i in tqdm(range(0, n, batch_size), desc=model_name.split('/')[-1]):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        pos[i:i+len(batch)] = probs[:, pi]
        neg[i:i+len(batch)] = probs[:, ni]
        if ui is not None:
            neu[i:i+len(batch)] = probs[:, ui]
        else:
            neu[i:i+len(batch)] = 1.0 - probs[:, pi] - probs[:, ni]

    df['finbert_neg']   = neg
    df['finbert_neu']   = neu
    df['finbert_pos']   = pos
    df['finbert_score'] = (pos - neg).astype(np.float32)

    print('\n  distribution:')
    print(df[['finbert_neg','finbert_neu','finbert_pos','finbert_score']].describe().round(4))
    strong = int((df['finbert_score'].abs() > 0.3).sum())
    print(f'  strong signal (|score|>0.3): {strong:,} ({100*strong/len(df):.1f}%)')

    df_out = df[['rep_event_id','title','finbert_neg','finbert_neu','finbert_pos','finbert_score']]
    df_out.to_parquet(output_path, index=False, compression='zstd')
    print(f'  saved      : {output_path}')

    # free GPU memory
    del model; torch.cuda.empty_cache()
    return df_out

## 2. 중국어: yiyanghkust/finbert-tone-chinese

In [ ]:
df_zh = run_finbert(
    model_name = 'yiyanghkust/finbert-tone-chinese',
    input_path = 'data/processed/finbert_input_zh.parquet',
    output_path= 'data/processed/finbert_results_zh.parquet',
    batch_size = 64
)

In [ ]:
print('=== ZH most NEGATIVE ===')
for _, r in df_zh.nsmallest(5, 'finbert_score').iterrows():
    print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')
print('\n=== ZH most POSITIVE ===')
for _, r in df_zh.nlargest(5, 'finbert_score').iterrows():
    print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')

## 3. 한국어: snunlp/KR-FinBert-SC

In [ ]:
df_ko = run_finbert(
    model_name = 'snunlp/KR-FinBert-SC',
    input_path = 'data/processed/finbert_input_ko.parquet',
    output_path= 'data/processed/finbert_results_ko.parquet',
    batch_size = 64
)

In [ ]:
print('=== KO most NEGATIVE ===')
for _, r in df_ko.nsmallest(5, 'finbert_score').iterrows():
    print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')
print('\n=== KO most POSITIVE ===')
for _, r in df_ko.nlargest(5, 'finbert_score').iterrows():
    print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')

## 4. 완료

다음 단계: 생성된 2개 파일을 로컬 `data/processed/`로 동기화한 뒤
```bash
python scripts/merge_v6.py
```
실행하여 v6 생성.